In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F


spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("seminar")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
sc = spark.sparkContext

Анализировать будет датасет отсюда https://www.kaggle.com/shelvigarg/credit-card-buyers

Definition

ID - Unique Identifier for a row

Gender - Gender of the Customer

Age - Age of the Customer (in Years)

Region_Code - Code of the Region for the customers

Occupation - Occupation Type for the customer

Channel_Code - Acquisition Channel Code for the Customer (Encoded)

Vintage - Vintage for the Customer (In Months)

Credit_Product - If the Customer has any active credit product (Home loan Personal loan, Credit Card etc.)

AvgAccountBalance - Average Account Balance for the Customer in last 12 Months

Is_Active - If the Customer is Active in last 3 Months

Загрузим данные и посмотрим, что там внутри

In [2]:
data = spark.read.csv("credit_card_data.csv", header=True, inferSchema=True)

In [3]:
data.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Region_Code: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Channel_Code: string (nullable = true)
 |-- Vintage: integer (nullable = true)
 |-- Credit_Product: string (nullable = true)
 |-- Avg_Account_Balance: integer (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Is_Lead: integer (nullable = true)



In [4]:
data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|
|HD3DSEMC|Female| 56|      RG268|Self_Employed|          X3|     26|            No|            1484315|      Yes|      0|
|BF3NC7KV|  Male| 34|      RG270|     Salaried|          X1|     19|            No|             470454|       No|      0|
|TEASRWXV|Female| 30|      RG282|     Salaried|          X1|     33|            No|             886787|       No|      0|
|ACUTYTWS|  Male| 56|   

Посмотрим различные базовые вещи

In [5]:
data.select([F.count(F.when(F.isnan(F.col(c).try_cast("double")) | F.col(c).isNull(), c)).alias(c) for c in data.columns]).show()

+---+------+---+-----------+----------+------------+-------+--------------+-------------------+---------+-------+
| ID|Gender|Age|Region_Code|Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|
+---+------+---+-----------+----------+------------+-------+--------------+-------------------+---------+-------+
|  0|     0|  0|          0|         0|           0|      0|         29325|                  0|        0|      0|
+---+------+---+-----------+----------+------------+-------+--------------+-------------------+---------+-------+



Пропуски только в кредитном продукте, логично заметь на тип, что кредита нет

In [6]:
data.select(F.col("Credit_Product")).groupBy("Credit_Product").count().show()

+--------------+------+
|Credit_Product| count|
+--------------+------+
|          NULL| 29325|
|            No|144357|
|           Yes| 72043|
+--------------+------+



In [7]:
data = data.fillna({"Credit_Product": "No"})

Проверим

In [8]:
data.select(F.col("Credit_Product")).groupBy("Credit_Product").count().show()

+--------------+------+
|Credit_Product| count|
+--------------+------+
|            No|173682|
|           Yes| 72043|
+--------------+------+



Посмотри на данные с точки зрения дисбаланса классов

In [9]:
data.count()

245725

In [10]:
data.select(F.col("Is_Lead")).groupBy("Is_Lead").count().withColumn("count", F.round(F.col("count") / data.count(), 2)).show()

+-------+-----+
|Is_Lead|count|
+-------+-----+
|      1| 0.24|
|      0| 0.76|
+-------+-----+



Ладно, достаточно, мы тут сейчас говорим про MLlib, всякие анализы - тема прошлого семинара

**Некоторые преобразования данных**

Начнем с простой обработки категориальных переменных

In [11]:
from pyspark.ml.feature import StringIndexer, IndexToString, OneHotEncoder

In [12]:
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
gender_indexer = gender_indexer.fit(data)
data = gender_indexer.transform(data)

data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|        1.0|
|HD3DSEMC|Female| 56|      RG268|Self_Employed|          X3|     26|            No|            1484315|      Yes|      0|        1.0|
|BF3NC7KV|  Male| 34|      RG270|     Salaried|          X1|     19|            No|             470454|       No|      0|        0.0|
|TEASRWXV|Female| 30|      RG282|     Salaried|          X1|  

In [13]:
gender_indexer.labels

['Male', 'Female']

Обратная трансформация доступна через метод

In [14]:
converter = IndexToString(inputCol="GenderIndex", outputCol="originalGender")
data = converter.transform(data)
data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|originalGender|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|        Female|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|        1.0|        Female|
|HD3DSEMC|Female| 56|      RG268|Self_Employed|          X3|     26|            No|            1484315|      Yes|      0|        1.0|        Female|
|BF3NC7KV|  Male| 34|      RG270|     Salaried|          X1|     19|            No|             470454|   

Давайте аналогично поступим с каналом продаж и типом занятости

In [15]:
occupation_indexer = StringIndexer(inputCol="Occupation", outputCol="OccupationIndex")
occupation_indexer = occupation_indexer.fit(data)
data = occupation_indexer.transform(data)

channel_indexer = StringIndexer(inputCol="Channel_Code", outputCol="ChannelIndex")
channel_indexer = channel_indexer.fit(data)
data = channel_indexer.transform(data)

data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|originalGender|OccupationIndex|ChannelIndex|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|        Female|            2.0|         1.0|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|        1.0|        Female|            1.0|         0.0|
|HD3DSEMC|Female| 56|      RG268|Self_Employed|          X3|     26|            No|            1484315|      Y

In [16]:
print(f"Occupation len = {len(occupation_indexer.labels)}, Channel_code len = {len(channel_indexer.labels)}")

Occupation len = 4, Channel_code len = 4


Тут по 4 категории, что самое простое, что приходит в голову? Правильно - OHE

In [17]:
ohe_encoder = OneHotEncoder(inputCols=["OccupationIndex", "ChannelIndex"], outputCols=["OccupationVector", "ChannelVec"])
ohe_encoder = ohe_encoder.fit(data)
data = ohe_encoder.transform(data)

data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|originalGender|OccupationIndex|ChannelIndex|OccupationVector|   ChannelVec|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|        Female|            2.0|         1.0|   (3,[2],[1.0])|(3,[1],[1.0])|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|        1.0|        Female|            1.0|

In [18]:
ohe_encoder.categorySizes

[4, 4]

Странный формат, не правда ли? Все из-за того, что тут у нас SparseVector

 На 4 категории нужен вектор размерности 3, а дальше храним позицию и 1 там, где нужная категория

In [19]:
data.select(F.col("OccupationVector")).head()

Row(OccupationVector=SparseVector(3, {2: 1.0}))

Теперь все надо собрать в одну структуру, чтобы можно было анализировать данные и строить модели

In [20]:
from pyspark.ml.feature import VectorAssembler

In [21]:
data.show(5)

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|originalGender|OccupationIndex|ChannelIndex|OccupationVector|   ChannelVec|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|        Female|            2.0|         1.0|   (3,[2],[1.0])|(3,[1],[1.0])|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             581988|       No|      0|        1.0|        Female|            1.0|

In [22]:
feature_columns = [
    "Age",
    "Vintage",
    "Avg_Account_Balance",
    "GenderIndex",
    "OccupationVector",
    "ChannelVec",
]

In [23]:
df_va = VectorAssembler(inputCols=feature_columns, outputCol="features")
data = df_va.transform(data)

In [24]:
data.printSchema()

root
 |-- ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Region_Code: string (nullable = true)
 |-- Occupation: string (nullable = true)
 |-- Channel_Code: string (nullable = true)
 |-- Vintage: integer (nullable = true)
 |-- Credit_Product: string (nullable = false)
 |-- Avg_Account_Balance: integer (nullable = true)
 |-- Is_Active: string (nullable = true)
 |-- Is_Lead: integer (nullable = true)
 |-- GenderIndex: double (nullable = false)
 |-- originalGender: string (nullable = true)
 |-- OccupationIndex: double (nullable = false)
 |-- ChannelIndex: double (nullable = false)
 |-- OccupationVector: vector (nullable = true)
 |-- ChannelVec: vector (nullable = true)
 |-- features: vector (nullable = true)



In [25]:
data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+--------------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|originalGender|OccupationIndex|ChannelIndex|OccupationVector|   ChannelVec|            features|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+--------------+---------------+------------+----------------+-------------+--------------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|        Female|            2.0|         1.0|   (3,[2],[1.0])|(3,[1],[1.0])|[73.0,43.0,104569...|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            

В полученном features можно автоматичеки проанализировать все переменные и если у кого-то уникальных значений меньше заданного вами порога, то они автоматичсеки переведутся в индексы при помощи pyspark.ml.feature import VectorIndexer

**Статистика**

В ml pyspark есть некоторые статистические методы, которые можно использовать для анализа

Корреляция

In [26]:
from pyspark.ml.stat import Correlation

In [27]:
corr = Correlation.corr(data, "features", method="pearson").collect()[0][0]

In [28]:
corr

DenseMatrix(10, 10, [1.0, 0.6312, 0.1452, -0.1521, 0.1527, -0.5632, 0.3948, -0.6646, ..., 0.0102, -0.116, 0.2933, -0.3238, 0.0005, -0.5272, -0.3843, 1.0], False)

In [29]:
corr.toArray()

array([[ 1.00000000e+00,  6.31242411e-01,  1.45232189e-01,
        -1.52075940e-01,  1.52651808e-01, -5.63226982e-01,
         3.94834177e-01, -6.64600051e-01,  4.56338440e-01,
         2.73153253e-01],
       [ 6.31242411e-01,  1.00000000e+00,  1.67433481e-01,
        -1.46379743e-01,  2.21023818e-01, -4.10109383e-01,
         1.55662661e-01, -5.71828453e-01,  5.38828562e-01,
         1.44931244e-01],
       [ 1.45232189e-01,  1.67433481e-01,  1.00000000e+00,
        -2.24772031e-02,  3.46714040e-03, -7.16906860e-02,
         6.03874569e-02, -9.81785292e-02,  1.06905544e-01,
         1.01634033e-02],
       [-1.52075940e-01, -1.46379743e-01, -2.24772031e-02,
         1.00000000e+00, -8.58626857e-02,  1.22439249e-01,
        -2.58175123e-02,  1.84372479e-01, -8.07817702e-02,
        -1.16018433e-01],
       [ 1.52651808e-01,  2.21023818e-01,  3.46714040e-03,
        -8.58626857e-02,  1.00000000e+00, -5.37283514e-01,
        -5.27660791e-01, -4.34990948e-01,  1.63662837e-01,
         2.

Можно вычислить корреляцию спирмена

In [30]:
corr = Correlation.corr(data, "features", method="spearman").collect()[0][0]
corr.toArray()

array([[ 1.00000000e+00,  6.52477119e-01,  1.78395617e-01,
        -1.66276503e-01,  2.50479037e-01, -5.94192974e-01,
         3.16280765e-01, -7.14324073e-01,  4.68523607e-01,
         3.11874367e-01],
       [ 6.52477119e-01,  1.00000000e+00,  1.98039737e-01,
        -1.37241231e-01,  2.26617372e-01, -4.03612329e-01,
         1.44248315e-01, -5.43361590e-01,  4.88850315e-01,
         1.86509197e-01],
       [ 1.78395617e-01,  1.98039737e-01,  1.00000000e+00,
        -3.35726184e-02,  1.74003362e-02, -9.65169583e-02,
         6.80005979e-02, -1.34528255e-01,  1.34941462e-01,
         2.41122172e-02],
       [-1.66276503e-01, -1.37241231e-01, -3.35726184e-02,
         1.00000000e+00, -8.58626857e-02,  1.22439249e-01,
        -2.58175123e-02,  1.84372479e-01, -8.07817702e-02,
        -1.16018433e-01],
       [ 2.50479037e-01,  2.26617372e-01,  1.74003362e-02,
        -8.58626857e-02,  1.00000000e+00, -5.37283514e-01,
        -5.27660791e-01, -4.34990948e-01,  1.63662837e-01,
         2.

Можно использовать хи-квадрат тест для оценки независимости каждой переменной в features относительно целевого признака, но этот тест для категориальных переменных, поэтому для примера на одной фиче

In [31]:
from pyspark.ml.stat import ChiSquareTest, KolmogorovSmirnovTest, Summarizer

In [32]:
r = ChiSquareTest.test(data, "OccupationVector", "Is_Lead")

In [33]:
r.show()

+--------------------+----------------+--------------------+
|             pValues|degreesOfFreedom|          statistics|
+--------------------+----------------+--------------------+
|[0.0,0.0,1.161583...|       [1, 1, 1]|[1420.86324574575...|
+--------------------+----------------+--------------------+



KS-тест

In [34]:
data.select(F.mean(F.col("Age")).alias("mean_Age"), F.stddev(F.col("Age")).alias("std_Age")).collect()

[Row(mean_Age=43.85630684708516, std_Age=14.828671804647998)]

In [35]:
ks = KolmogorovSmirnovTest.test(data, "Age", "norm", 44, 15).first()

In [36]:
ks

Row(pValue=2.045950076023928e-10, statistic=0.12561207843265515)

Еще можно посчитать разные статистики

In [37]:
summarizer = Summarizer.metrics("mean", "count")
data.select(summarizer.summary(data.features)).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|aggregate_metrics(features, 1.0)                                                                                                                                                                           |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|{[43.85630684708497,46.959141316512564,1128403.1010194393,0.4538732322718486,0.4105646556109472,0.29300640960423235,0.2855753382846678,0.42208973445925324,0.2796296673110184,0.27561705158205313], 245725}|
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

**Работа с фичами**

Квантизация

In [38]:
from pyspark.ml.feature import QuantileDiscretizer

Обучаем

In [39]:
discretizer = QuantileDiscretizer(numBuckets=5, inputCol="Age", outputCol="Age_quant")
discretizer = discretizer.fit(data)

In [40]:
data = discretizer.transform(data)

In [41]:
(
    data.select("Age", "Age_quant")
    .groupby("Age_quant")
    .agg(
        F.min("Age").alias("min_age"),
        F.max("Age").alias("max_age"),
        F.count("Age").alias("count")
    )
    .orderBy("Age_quant")
    .show(5)
)

+---------+-------+-------+-----+
|Age_quant|min_age|max_age|count|
+---------+-------+-------+-----+
|      0.0|     23|     28|43790|
|      1.0|     29|     35|52017|
|      2.0|     36|     46|46007|
|      3.0|     47|     55|50808|
|      4.0|     56|     85|53103|
+---------+-------+-------+-----+



Заполнить пропуски можно через Imputer

Заполнять пропуски умеет только для числовых переменных, поэтому попробуем на игрушечном примере



In [42]:
from pyspark.ml.feature import Imputer

In [43]:
df = spark.createDataFrame(
    [
        (1.0, float("nan")),
        (2.0, float("nan")),
        (float("nan"), 3.0),
        (4.0, 4.0),
        (5.0, 5.0),
    ],
    ["a", "b"],
)

# стратегия может быть 'mean', 'median', 'mode'
# через setMissingValue(0.0) можно сказать, что пропуски - это 0
imputer = Imputer(inputCols=["a", "b"], outputCols=["out_a", "out_b"], strategy="mean")
imputer = imputer.fit(df)
imputer.transform(df).show()

+---+---+-----+-----+
|  a|  b|out_a|out_b|
+---+---+-----+-----+
|1.0|NaN|  1.0|  4.0|
|2.0|NaN|  2.0|  4.0|
|NaN|3.0|  3.0|  3.0|
|4.0|4.0|  4.0|  4.0|
|5.0|5.0|  5.0|  5.0|
+---+---+-----+-----+



**Pipeline**

Как и в scikit-learn можно создавать пайплайны обработки данных

Мы много делали преобразований, давайте соберем все в 1 пайплайн

In [44]:
from pyspark.ml import Pipeline

In [45]:
# string в индесы
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
occupation_indexer = StringIndexer(inputCol="Occupation", outputCol="OccupationIndex")
channel_indexer = StringIndexer(inputCol="Channel_Code", outputCol="ChannelIndex")

# OHE
ohe_encoder = OneHotEncoder(inputCols=["OccupationIndex", "ChannelIndex"], outputCols=["OccupationVector", "ChannelVec"])

# квантизация
discretizer = QuantileDiscretizer(numBuckets=5, inputCol="Age", outputCol="Age_quant")

# собираем все в вектор
feature_columns = [
    "Age",
    "Vintage",
    "Avg_Account_Balance",
    "GenderIndex",
    "OccupationVector",
    "ChannelVec",
    "Age_quant",
]
vector_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

собираем все в пайплайн

In [46]:
pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    channel_indexer,
    ohe_encoder,
    discretizer,
    vector_assembler,
])

Давайте заново загрузим данные и сделаем трансформацию

---



In [47]:
data = spark.read.csv("credit_card_data.csv", header=True, inferSchema=True)
data = data.fillna({"Credit_Product": "No"})
pipeline = pipeline.fit(data)

In [48]:
transformed_data = pipeline.transform(data)

In [49]:
transformed_data.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+---------------+------------+----------------+-------------+---------+--------------------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|OccupationIndex|ChannelIndex|OccupationVector|   ChannelVec|Age_quant|            features|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+---------------+------------+----------------+-------------+---------+--------------------+
|NNVBBKZB|Female| 73|      RG268|        Other|          X3|     43|            No|            1045696|       No|      0|        1.0|            2.0|         1.0|   (3,[2],[1.0])|(3,[1],[1.0])|      4.0|[73.0,43.0,104569...|
|IDD62UNG|Female| 30|      RG277|     Salaried|          X1|     32|            No|             5819

In [50]:
transformed_data.select("Is_Lead", "features").show(5)

+-------+--------------------+
|Is_Lead|            features|
+-------+--------------------+
|      0|[73.0,43.0,104569...|
|      0|[30.0,32.0,581988...|
|      0|[56.0,26.0,148431...|
|      0|(11,[0,1,2,5,7,10...|
|      0|[30.0,33.0,886787...|
+-------+--------------------+
only showing top 5 rows


**Модельки**

Пора нам уже что-то обучить, начнем с логрега

In [51]:
from pyspark.ml.classification import LogisticRegression, LogisticRegressionModel

In [52]:
lr = LogisticRegression(featuresCol="features", labelCol="Is_Lead", predictionCol="prediction", maxIter=100, probabilityCol="proba")
lr = lr.fit(transformed_data)

Сохраним

In [53]:
lr.write().overwrite().save("logreg_model")

Загрузка

In [54]:
lr2 = LogisticRegressionModel.load("logreg_model")

Коэффициенты и метрики

In [55]:
print(f"Coefficients: {lr.coefficients}")
print(f"Intercept: {lr.intercept}")

Coefficients: [-0.016713044807990633,0.009916392983826688,9.918267490402371e-09,-0.05296577197133461,-1.5152849582664052,-0.5266676981657711,-1.5009517711157354,-1.5679772035561022,-0.14595364437537453,-0.05760664452060193,0.34189553941664735]
Intercept: 0.09853176624050297


In [56]:
print(f"Coefficients: {lr2.coefficients}")
print(f"Intercept: {lr2.intercept}")

Coefficients: [-0.016713044807990633,0.009916392983826688,9.918267490402371e-09,-0.05296577197133461,-1.5152849582664052,-0.5266676981657711,-1.5009517711157354,-1.5679772035561022,-0.14595364437537453,-0.05760664452060193,0.34189553941664735]
Intercept: 0.09853176624050297


In [57]:
print(f"ROC_AUC = {lr.summary.areaUnderROC}")

ROC_AUC = 0.7291589501476651


In [58]:
lr.summary.recallByLabel

[0.9880226422744709, 0.10472138347515783]

In [59]:
lr.params

[Param(parent='LogisticRegression_1040e0788e88', name='aggregationDepth', doc='suggested depth for treeAggregate (>= 2).'),
 Param(parent='LogisticRegression_1040e0788e88', name='elasticNetParam', doc='the ElasticNet mixing parameter, in range [0, 1]. For alpha = 0, the penalty is an L2 penalty. For alpha = 1, it is an L1 penalty.'),
 Param(parent='LogisticRegression_1040e0788e88', name='family', doc='The name of family which is a description of the label distribution to be used in the model. Supported options: auto, binomial, multinomial'),
 Param(parent='LogisticRegression_1040e0788e88', name='featuresCol', doc='features column name.'),
 Param(parent='LogisticRegression_1040e0788e88', name='fitIntercept', doc='whether to fit an intercept term.'),
 Param(parent='LogisticRegression_1040e0788e88', name='labelCol', doc='label column name.'),
 Param(parent='LogisticRegression_1040e0788e88', name='lowerBoundsOnCoefficients', doc='The lower bounds on coefficients if fitting under bound cons

In [60]:
lr.transform(transformed_data.select("Is_Lead", "features")).show()

+-------+--------------------+--------------------+--------------------+----------+
|Is_Lead|            features|       rawPrediction|               proba|prediction|
+-------+--------------------+--------------------+--------------------+----------+
|      0|[73.0,43.0,104569...|[1.01703314359247...|[0.73439428907074...|       0.0|
|      0|[30.0,32.0,581988...|[1.88547782413311...|[0.86823905458581...|       0.0|
|      0|[56.0,26.0,148431...|[0.91147290916398...|[0.71330147094143...|       0.0|
|      0|(11,[0,1,2,5,7,10...|[2.02938356422976...|[0.88384780907754...|       0.0|
|      0|[30.0,33.0,886787...|[1.87253835313648...|[0.86675171295998...|       0.0|
|      0|(11,[0,1,2,4,7,10...|[2.23035701748805...|[0.90294265153482...|       0.0|
|      1|(11,[0,1,2,6,8,10...|[1.00819128083242...|[0.73266603083259...|       0.0|
|      0|[48.0,13.0,444724...|[1.25888814052507...|[0.77883464798460...|       0.0|
|      0|[40.0,38.0,127428...|[1.12259469904801...|[0.75446968923717...|    

**Подбор параметров**

Тут нет всяких hyperopt, optuna...есть стандартная кросс-валидация и поиск по сетке

In [61]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator

Для этого соберем все в пайплайн. Можно было "вложить" старый пайплайн в новый, но соберем все с самого начала

In [62]:
# string в индесы
gender_indexer = StringIndexer(inputCol="Gender", outputCol="GenderIndex")
occupation_indexer = StringIndexer(inputCol="Occupation", outputCol="OccupationIndex")
channel_indexer = StringIndexer(inputCol="Channel_Code", outputCol="ChannelIndex")

# OHE
ohe_encoder = OneHotEncoder(inputCols=["OccupationIndex", "ChannelIndex"], outputCols=["OccupationVector", "ChannelVec"])

# квантизация
discretizer = QuantileDiscretizer(numBuckets=5, inputCol="Age", outputCol="Age_quant")

# собираем все в вектор
feature_columns = [
    "Age",
    "Vintage",
    "Avg_Account_Balance",
    "GenderIndex",
    "OccupationVector",
    "ChannelVec",
    "Age_quant",
]
vector_assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

lr = LogisticRegression(featuresCol="features", labelCol="Is_Lead", predictionCol="prediction", maxIter=100, probabilityCol="proba")

In [63]:
pipeline = Pipeline(stages=[
    gender_indexer,
    occupation_indexer,
    channel_indexer,
    ohe_encoder,
    discretizer,
    vector_assembler,
    lr,
])

Сетка параметров

In [64]:
paramGrid = ParamGridBuilder().addGrid(discretizer.numBuckets, [5, 10]).addGrid(lr.maxIter, [10, 20]).build()

Разобьем данные на train, test

In [65]:
train, test = data.randomSplit([0.7, 0.3], seed=7)

Описываем стратегию кросс-валидации

In [66]:
crossval = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="Is_Lead", metricName="areaUnderROC"),
    numFolds=2,
    parallelism=2,
)

Гоняем сетку. Знаю, перебор по сетке прошлый век, но что поделать)

In [67]:
cvModel = crossval.fit(train)

In [68]:
cvModel.avgMetrics

[np.float64(0.7287462430144251),
 np.float64(0.7292052202464849),
 np.float64(0.7286701219711768),
 np.float64(0.7293428335349428)]

Параметры

In [69]:
import numpy as np
print(cvModel.getEstimatorParamMaps()[np.argmax(cvModel.avgMetrics)])

{Param(parent='QuantileDiscretizer_3f3b88f3a2fe', name='numBuckets', doc='Maximum number of buckets (quantiles, or categories) into which data points are grouped. Must be >= 2.'): 10, Param(parent='LogisticRegression_139ed91edce7', name='maxIter', doc='max number of iterations (>= 0).'): 20}


Сделаем предикт

In [70]:
test_pred = cvModel.transform(test)

In [71]:
test_pred.show()

+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+---------------+------------+----------------+-------------+---------+--------------------+--------------------+--------------------+----------+
|      ID|Gender|Age|Region_Code|   Occupation|Channel_Code|Vintage|Credit_Product|Avg_Account_Balance|Is_Active|Is_Lead|GenderIndex|OccupationIndex|ChannelIndex|OccupationVector|   ChannelVec|Age_quant|            features|       rawPrediction|               proba|prediction|
+--------+------+---+-----------+-------------+------------+-------+--------------+-------------------+---------+-------+-----------+---------------+------------+----------------+-------------+---------+--------------------+--------------------+--------------------+----------+
|224VSEND|  Male| 29|      RG261|     Salaried|          X1|     13|            No|             736866|      Yes|      0|        0.0|            1.0|         0.0|   (

Проверим модель

In [72]:
evaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="Is_Lead", metricName="areaUnderROC")

In [73]:
evaluator.evaluate(test_pred)

0.7279221958853093

Сохраним пайплайн

In [74]:
cvModel.write().overwrite().save("model")

вместо кросс-валидации можно взять TrainValidationSplit для подбора параметров, это train_test_split